In [4]:
import re
import csv
import os

def convert_log_to_csv_simplified(input_filename="平均クラスター係数基準/お互いにDP(lowdense_cc_user_p2_egotwitter).txt", output_filename="平均クラスター係数基準/お互いにDP(lowdense_cc_user_p2_egotwitter).csv"):
    """
    ログファイルから最適化データを抽出し、指定された項目でCSVファイルとして保存する
    """
    
    # 最終的なデータ格納リスト
    data_rows = []
    
    # CSVのヘッダー行 (実行回数とDP_timeを除外)
    header = [
        "Seed", 
        "ありの解", "ありの解2", 
        "ありをなし", 
        "なしの解", 
        "なしの解をアリ", "なしの解をアリ2"
    ]
    
    # 現在の処理状態を保持する変数
    current_seed = None
    
    # 正規表現パターン
    seed_pattern = re.compile(r"^seed (\d+)")
    start_dp_pattern = re.compile(r"^start_DP") # 新しい実行ブロックの検出用
    cost_sum_pattern = re.compile(r"^cost_sum: (\d+)")
    
    # データを抽出するためのパターン
    data_patterns = [
        ("あり", re.compile(r"虚偽情報アリの解 \[.*?\] (\d+\.\d+) (\d+\.\d+)")),
        ("あり_なし", re.compile(r"虚偽情報アリの解を無しに使ってみたら\.\.\. (\d+\.\d+) 0")),
        ("なし", re.compile(r"虚偽情報なしの解 \[.*?\] (\d+\.\d+) 0")),
        ("なし_あり", re.compile(r"虚偽情報なしの解をアリに使ってみたら\.\.\. (\d+\.\d+) (\d+\.\d+)"))
    ]

    try:
        with open(input_filename, 'r', encoding='utf-8') as f:
            lines = f.readlines()
    except FileNotFoundError:
        print(f"エラー: 入力ファイル '{input_filename}' が見つかりません。")
        return

    temp_data = {}

    for line in lines:
        line = line.strip()

        # 1. seedを検出
        m_seed = seed_pattern.match(line)
        if m_seed:
            current_seed = m_seed.group(1)
            continue
        
        # 2. 新しい実行ブロックの開始を検出
        if start_dp_pattern.match(line):
            temp_data = {"Seed": current_seed}
            continue

        # 3. cost_sumを検出
        m_cost_sum = cost_sum_pattern.match(line)
        if m_cost_sum:
            temp_data["cost_sum"] = m_cost_sum.group(1)
            continue
            
        # 4. コストデータを検出
        for key_prefix, pattern in data_patterns:
            m_data = pattern.match(line)
            if m_data and temp_data: # temp_dataが空でないことを確認
                
                if key_prefix == "あり":
                    temp_data["ありの解"] = m_data.group(1)
                    temp_data["ありの解2"] = m_data.group(2)
                elif key_prefix == "あり_なし":
                    temp_data["ありをなし"] = m_data.group(1)
                elif key_prefix == "なし":
                    temp_data["なしの解"] = m_data.group(1)
                elif key_prefix == "なし_あり":
                    temp_data["なしの解をアリ"] = m_data.group(1)
                    temp_data["なしの解をアリ2"] = m_data.group(2)
                    
                    # 'なし_あり'の行で一つの実行ブロックが完結と見なす
                    # 全ての情報が揃ったと仮定してデータ行を作成し、リストに追加
                    data_rows.append([temp_data.get(h, '') for h in header])
                    temp_data = {"Seed": current_seed} # 次のブロックのためにシードを保持し、他をクリア

    # 5. CSVファイルに書き込み
    with open(output_filename, 'w', newline='', encoding='utf-8') as outfile:
        writer = csv.writer(outfile)
        writer.writerow(header)
        writer.writerows(data_rows)
        
    print(f"✅ データの抽出と変換が完了しました。")
    print(f"CSVファイル: '{output_filename}' として保存されました。")
    print(f"保存場所: {os.path.abspath(output_filename)}")

# 実行
if __name__ == "__main__":
    # 元のテキストファイル名を 'input.txt' に変更するか、ここを書き換えてください
    convert_log_to_csv_simplified("平均クラスター係数基準/お互いにDP(lowdense_cc_user_p2_egotwitter).txt", "平均クラスター係数基準/お互いにDP(lowdense_cc_user_p2_egotwitter).csv")

✅ データの抽出と変換が完了しました。
CSVファイル: '平均クラスター係数基準/お互いにDP(lowdense_cc_user_p2_egotwitter).csv' として保存されました。
保存場所: /Users/kaoriogawa/研究/community-detection-social-networks/result/平均クラスター係数基準/お互いにDP(lowdense_cc_user_p2_egotwitter).csv
